In [18]:
from dotenv import load_dotenv
load_dotenv()
import os 

PROJECT_ABSOLUTE_PATH = os.getenv("PROJECT_ABSOLUTE_PATH")

In [19]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-20b',temperature=0.1)

In [20]:
from fastmcp import Client, ClientGroup
from fastmcp.client.transports import StdioTransport
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
from typing import Annotated

In [21]:
exchange_rate_transport = StdioTransport(
    command=f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/.venv/bin/python",
    args=[
        f"{PROJECT_ABSOLUTE_PATH}/ai-engineering-learning/week-6/practice/exchange_rate_conversion_server.py"
    ]
)

exchange_rate_client = Client(transport=exchange_rate_transport)

In [24]:
async with MCPAdapter(clients) as adapter:
    tools = await adapter.list_tools()

    llm_with_tools = llm.bind_tools(tools)

    graph = StateGraph(MessagesState)

    graph.add_node("chat_node", chat_node)
    graph.add_node("tools", ToolNode(tools))

    graph.add_edge(START, "chat_node")

    graph.add_conditional_edges(
        "chat_node",
        tools_condition
    )

    graph.add_edge("tools", "chat_node")

    workflow = graph.compile()

    result = await workflow.ainvoke({
        "messages": [
            HumanMessage(
                content="What is the currency conversion rate between USD and INR?"
            )
        ]
    })

    print(result["messages"][-1].content)

The current conversion rate is **1 USD = 96.3495 INR**.
